In [1]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

# The 5 universal API components
# 1. BASE URL
# 2. AUTH KEY
# 3. AUTH HEADER
# 4. MODEL STRING
# 5. MESSAGES LIST

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "openai/gpt-oss-120b"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello!


In [2]:
# 3. THE 5 UNIVERSAL API COMPONENTS

# 1. BASE URL — address of the model server
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY — proves you are authorised
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "openai/gpt-oss-120b",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send and parse
response = requests.post(BASE_URL, headers=headers, json=payload)
response.raise_for_status()
data = response.json()

print(data["choices"][0]["message"]["content"])

Hello! How can I assist you today?


In [3]:
# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "openai/gpt-oss-120b",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

full_call.raise_for_status()
data = full_call.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))

Answer: The capital of France is **Paris**.
Tokens used: 125


In [4]:
# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL, default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(self.base_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)

In [5]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 2 + 2?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Hello Alex!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 2 + 2 = 4.

Multi-turn: Your name is Alex.


In [6]:
# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Alex"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE: Activity data from Ep1, embeddings from Ep2, attention from Ep3, prompting strategy from Ep4.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)

Based on the information I have, I can only generate a recommendation that directly reflects your recorded activity data, embeddings, and attention patterns. At this moment I don’t have a recent activity entry or goal context to reference, so I can’t produce a specific next‑step suggestion.

Please share the latest entry from your activity log (e.g., the most recent task you completed, the current project you’re tracking, or the goal you’re working toward). With that data I can give you a precise, data‑driven recommendation for what to do next.


In [7]:
# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))

I can’t generate a data‑driven priority list without a recent activity record or a defined goal to reference. Please provide the latest entry from your activity log (e.g., the most recent task you completed, the project you’re currently focused on, or the specific objective you’re tracking). With that information I’ll produce a concise, three‑item priority list tailored to your actual data.


In [9]:
# Context for digital twin

user_name = "Lynton"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE:
  User Background: Senior Data Architect with 12+ years in cloud infrastructure, distributed systems, and enterprise data modeling.
  Core Tech Stack: Advanced proficiency in Python, Rust, SQL, Apache Spark, Snowflake, Databricks, AWS (S3, Redshift, EKS), and Terraform.
  Preferences & Workstyle:
    Prioritizes high-throughput, low-latency architecture designs.
    Prefers CLI over GUI; heavily relies on Neovim, Tmux, and customized zsh scripts.
    Strong bias for modular, test-driven infrastructure as code (IaC).
    Strict adherence to cost-optimization and data security protocols (GDPR, SOC 2).
  Active Projects & Focus:
    Migrating legacy monolithic data warehouses to a decentralized Data Mesh architecture.
    Evaluating real-time streaming pipelines using Apache Flink vs. Kafka Streams.
    Standardizing internal API schemas using OpenAPI 3.0 and gRPC.
  Historical Interactions & Decisions:
    Consistently rejects non-deterministic ML frameworks in favor of explainable statistical models for core business metrics.
    Rejects unstructured documentation; favors concise, version-controlled Markdown specs stored alongside codebase.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

print("-------------------------------------")
twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))
print("-------------------------------------")

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))
print("-------------------------------------")

-------------------------------------
**Next‑action roadmap (high‑throughput, low‑latency, cost‑optimized, GDPR‑compliant)**  

| Priority | Area | Concrete CLI‑driven step | Rationale |
|----------|------|--------------------------|-----------|
| **1** | **Data‑Mesh domain definition** | 1. Open a new `data-mesh/domain‑catalog.md` in your repo. <br>2. List all legacy warehouse tables and map each to a *domain* (e.g., `customer`, `product`, `finance`). <br>3. For each domain, draft a **Data Product Contract** (schema, SLA, ownership) in Markdown and version‑control it. | Establishes clear ownership and contract‑first boundaries before any infrastructure is provisioned. |
| **2** | **Infrastructure as Code (IaC) scaffolding** | 1. Create a Terraform module `modules/data-mesh` that provisions: <br>   - AWS S3 buckets (encrypted, versioned) per domain <br>   - IAM roles/policies scoped to domain owners <br>   - Glue Catalog databases & tables (or Snowflake schemas) <br>   - Optional Redsh